# Weighting and Rank Sensitivity

**DS4DH · Module 08 — Index and Metric Design**

*Technique:* Treating weights as a value judgement and testing how much they matter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/08c_weighting_sensitivity.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# This notebook reads the CSVs sitting next to it. In Colab, upload them from
# the pack's data/ folder when prompted. The exists() guard means a re-run
# part-way through a session will not ask you to upload all over again.
NEEDED = ['merged_dataset.csv']
missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    try:
        from google.colab import files
        print('Upload from the data/ folder of the pack: ' + ', '.join(missing))
        files.upload()
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Equal weights feel neutral. They are not.

Weighting three components at one third each is a claim that renter burden,
renter income and the tenure gap matter equally for housing accessibility. That
is a substantive position, and someone at a tenant advocacy organisation would
reasonably disagree with it.

Since you cannot avoid taking a position, the professional move is to show how
much the answer depends on it.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
HAI_COLS = ['Renter', 'rent_income', 'renter_owner_gap']
hai_df = base.dropna(subset=HAI_COLS).copy()

def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

hai_df['n_burden'] = minmax(hai_df['Renter'])
hai_df['n_income'] = 1 - minmax(hai_df['rent_income'])
hai_df['n_gap'] = minmax(hai_df['renter_owner_gap'])

print(f'{len(hai_df)} CSDs with all three components')

In [ ]:
SCHEMES = {
    'equal':          {'n_burden': 1 / 3, 'n_income': 1 / 3, 'n_gap': 1 / 3},
    'tenant-focused': {'n_burden': 0.55, 'n_income': 0.35, 'n_gap': 0.10},
    'income-focused': {'n_burden': 0.20, 'n_income': 0.60, 'n_gap': 0.20},
    'gap-focused':    {'n_burden': 0.20, 'n_income': 0.20, 'n_gap': 0.60},
}

for name, w in SCHEMES.items():
    hai_df[name] = sum(hai_df[c] * wt for c, wt in w.items())
    hai_df[f'rank_{name}'] = hai_df[name].rank(ascending=False)

print('Top 5 least-accessible CSDs under each weighting:')
print()
for name in SCHEMES:
    top = hai_df.nsmallest(5, f'rank_{name}')['geography_name'].str[:28].tolist()
    print(f'{name:<16}{" | ".join(t[:18] for t in top[:3])}')

## How much does the ranking actually move?

Two measures: rank correlation between schemes (how similar overall), and the
largest single move (how badly one place can be affected).

In [ ]:
names = list(SCHEMES)
print('Spearman rank correlation between weighting schemes:')
print()
print(f'{"":<16}' + ''.join(f'{n[:14]:>16}' for n in names))
for a in names:
    row = f'{a:<16}'
    for b in names:
        rho = stats.spearmanr(hai_df[f'rank_{a}'], hai_df[f'rank_{b}']).statistic
        row += f'{rho:>16.3f}'
    print(row)

In [ ]:
hai_df['rank_shift'] = (hai_df['rank_equal'] - hai_df['rank_tenant-focused']).abs()

print(f'equal vs tenant-focused, {len(hai_df)} CSDs:')
print(f'  median rank shift : {hai_df["rank_shift"].median():.0f} positions')
print(f'  90th percentile   : {hai_df["rank_shift"].quantile(0.9):.0f} positions')
print(f'  largest shift     : {hai_df["rank_shift"].max():.0f} positions')
print()
print('Most affected places:')
cols = ['geography_name', 'cma', 'rank_equal', 'rank_tenant-focused', 'rank_shift']
print(hai_df.nlargest(6, 'rank_shift')[cols].to_string(index=False))

In [ ]:
fig, ax = plt.subplots()
ax.scatter(hai_df['rank_equal'], hai_df['rank_tenant-focused'], s=22, alpha=0.6)
lim = [0, len(hai_df)]
ax.plot(lim, lim, color='#E8663D', ls='--', label='unchanged rank')
ax.set_xlabel('rank under equal weights')
ax.set_ylabel('rank under tenant-focused weights')
ax.set_title('Points far from the line are places whose story depends on the weights')
ax.legend()
plt.tight_layout()
plt.show()

## Testing the top of the table specifically

Overall rank correlation can be high while the *top ten* — the part that ends up
in a press release — churns completely. That is the number to report.

In [ ]:
TOP_N = 10
sets = {n: set(hai_df.nsmallest(TOP_N, f'rank_{n}')['csd_code']) for n in names}

print(f'Overlap in the top {TOP_N} between schemes:')
print()
print(f'{"":<16}' + ''.join(f'{n[:14]:>16}' for n in names))
for a in names:
    row = f'{a:<16}'
    for b in names:
        row += f'{len(sets[a] & sets[b]):>16}'
    print(row)
print()
stable = set.intersection(*sets.values())
print(f'{len(stable)} of {TOP_N} CSDs appear in the top {TOP_N} under ALL four schemes.')
print('Those are the ones you can name without a caveat about weighting.')

### 🔧 Your turn 1

Add a fifth scheme that puts all the weight on one component, e.g.
`{'n_burden': 1.0, 'n_income': 0.0, 'n_gap': 0.0}`.

How many of the stable top-10 survive? A place that stays at the top even under a
single-component index is a genuinely robust finding.

### 🔧 Your turn 2

Change `TOP_N` to 3 and to 25.

Is the overlap proportionally better or worse at the very top? What does that say
about quoting "the worst municipality for housing accessibility" as a headline?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Typically only a handful survive a single-component index, and
those are places that are extreme on every dimension at once. Those are the ones
to name in a brief. For the rest, the honest phrasing is comparative and hedged:
*"among the least accessible under a range of reasonable weightings"*.

**Your turn 2.** Overlap is proportionally *worse* at the very top. The top 3 is
the least stable part of the ranking, because small differences in composite score
separate places that are all extreme. This is the direct argument against
single-place headlines: "the worst municipality" is the claim your method supports
least well, and it is the one a press office will want most. Report a band, not a
winner.

</details>

## Where this stops

Module 08's discipline in one line: publish the weights, publish the sensitivity,
and name only the places that survive it.

Next: Module 09 asks where these places are — without a map.